In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime, time
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import zipfile
import io
import statsmodels.api as sm
import os
import glob
from sqlalchemy import create_engine, types as satypes
import contextlib
import xml.etree.ElementTree as ET
from tqdm import tqdm

In [9]:
folder_path = os.path.join(os.path.expanduser("~"), "Desktop")
os.path.samefile(folder_path, os.getcwd()), time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

(True, '2025-10-09 19:33:46')

In [13]:
engine = create_engine(f"postgresql+psycopg://{username}:{password}@{host}:{port}/{database}")
connection = engine.connect()

In [15]:
sp500_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

try:
    wikipedia_tables = pd.read_html(sp500_url)
except Exception:
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(sp500_url, headers=headers)
    response.raise_for_status()
    wikipedia_tables = pd.read_html(io.StringIO(response.text))

wikipedia_table = wikipedia_tables[0]

wikipedia_table = wikipedia_tables[0].rename(columns={
    "Symbol": "ticker",
    "Security": "company_name",
    "GICS Sector": "sector",
    "GICS Sub-Industry": "sub_industry",
    "Headquarters Location": "headquarters_location",
    "Date added": "date_added",
    "CIK": "cik",
    "Founded": "founded"
})

wikipedia_table.to_sql("sp500_reference", engine, if_exists="replace", index=False)

-1

In [16]:
sp500_reference = pd.read_sql("SELECT * FROM sp500_reference;", engine)
sp500_reference

,ticker,company_name,sector,sub_industry,headquarters_location,date_added,cik,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [17]:
symbol_column = sp500_reference["ticker"]

ticker_list = []
for ticker in symbol_column:
    ticker_list.append(ticker)

ticker_list = [ticker.replace(".", "-") for ticker in ticker_list]  # Wikipedia uses "." in some tickers to denote class A/B shares. Also this is an example of list comprehension.
ticker_list.sort()

In [18]:
with engine.connect() as conn:
    ohlcv_dates = pd.read_sql("""
        SELECT 
            MIN(date) AS earliest_date,
            MAX(date) AS latest_date
        FROM sp500_ohlcv;
    """, conn).iloc[0]

    edgar_dates = pd.read_sql("""
        SELECT 
            MIN(filed) AS earliest_date,
            MAX(filed) AS latest_date
        FROM sp500_edgar_financials;
    """, conn).iloc[0]

ohlcv_start = pd.to_datetime(ohlcv_dates["earliest_date"])
ohlcv_end   = pd.to_datetime(ohlcv_dates["latest_date"])
edgar_start = pd.to_datetime(edgar_dates["earliest_date"])
edgar_end   = pd.to_datetime(edgar_dates["latest_date"])

ohlcv_start, ohlcv_end, edgar_start, edgar_end

(Timestamp('2025-01-02 00:00:00'),
 Timestamp('2025-10-08 00:00:00'),
 Timestamp('2009-04-15 00:00:00'),
 Timestamp('2025-10-09 00:00:00'))

In [19]:
start_input = input("Start date (YYYY-MM-DD or 'start'): ").lower()
if start_input == "start":
    start = ohlcv_start
else:
    start = pd.Timestamp(datetime.datetime.strptime(start_input, "%Y-%m-%d")).normalize()
end_input = input("End date (YYYY-MM-DD or 'now'): ").lower()
if end_input == "now":
    end = pd.Timestamp.now().normalize()
else:
    end = pd.Timestamp(datetime.datetime.strptime(end_input, "%Y-%m-%d")).normalize()
    
start, end

(Timestamp('2025-01-02 00:00:00'), Timestamp('2025-10-09 00:00:00'))

In [20]:
sp500df = pd.read_sql("SELECT * FROM sp500_ohlcv;",
                      engine, parse_dates=["date"]).pivot(index="date",
                                                          columns="ticker",
                                                          values=["open","high", "low", "close", "adj_close", "volume"]).sort_index(axis=1, level=0)
sp500df.tail()

adj_close                                                  \
ticker               A        AAPL        ABBV        ABNB         ABT   
date                                                                     
2025-10-02  138.699997  257.130005  236.559998  121.489998  132.990005   
2025-10-03  141.639999  258.019989  233.910004  120.220001  134.589996   
2025-10-06  141.610001  256.690002  230.190002  120.349998  133.740005   
2025-10-07  138.559998  256.480011  232.830002  119.849998  133.020004   
2025-10-08  140.809998  258.059998  231.240005  119.989998  134.270004   

                                                                      ...  \
ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
date                                                                  ...   
2025-10-02  89.080002  244.339996  351.480011  241.669998  59.110001  ...   
2025-10-03  90.790001  245.320007  346.739990  241.990005  61.040001  ...   
2025-10-06  91.349998  248.169998  350.140015  242.500000  62.450001  ...   
2025-10-07  94.099998  251.229996  348.309998  233.750000  62.889999  ...   
2025-10-08  93.099998  252.979996  348.769989  237.929993  62.220001  ...   

               volume                                                          \
ticker             WY       WYNN        XEL         XOM        XYL        XYZ   
date                                                                            
2025-10-02  3592700.0  1286100.0  6982500.0  13059400.0  1347100.0  7852600.0   
2025-10-03  2943500.0  3619500.0  3850200.0  12948600.0  1245600.0  5755700.0   
2025-10-06  3680500.0  1799500.0  4082600.0  12034200.0  1330600.0  5318100.0   
2025-10-07  4592700.0  1443800.0  3715700.0  11945000.0  1190800.0  5428000.0   
2025-10-08  3475100.0  2316000.0  3142800.0  12299100.0  1029500.0  5993900.0   

                                                       
ticker            YUM        ZBH      ZBRA        ZTS  
date                                                   
2025-10-02  1800700.0   730500.0  450100.0  3262300.0  
2025-10-03  1288500.0   896200.0  452300.0  2569700.0  
2025-10-06  1395000.0  1065500.0  558200.0  3114900.0  
2025-10-07  1657100.0   920800.0  476800.0  2753700.0  
2025-10-08  1150600.0  1100700.0  577800.0  3016500.0  

[5 rows x 3048 columns]

In [21]:
sp500df_ohlcv_append_raw = yf.download(ticker_list,start=sp500df.index.max()+ pd.Timedelta(days=1), end=end, group_by="ticker", auto_adjust=False, threads=False)

[*********************100%***********************]  503 of 503 completed

503 Failed downloads:
['AMP', 'FDS', 'LW', 'CHTR', 'WMB', 'STX', 'TPL', 'CI', 'EXE', 'ANET', 'GPC', 'DLTR', 'IP', 'GEHC', 'NRG', 'MCD', 'UDR', 'AZO', 'URI', 'APH', 'DOC', 'NEM', 'ES', 'DXCM', 'TXN', 'DOV', 'RTX', 'LUV', 'RL', 'IT', 'INVH', 'AVB', 'CDNS', 'OMC', 'MAA', 'PM', 'CB', 'PSX', 'EQR', 'VTRS', 'IVZ', 'BLDR', 'ZTS', 'SCHW', 'GPN', 'LYB', 'HPQ', 'APTV', 'LULU', 'MDT', 'EXR', 'WST', 'VICI', 'DHR', 'RMD', 'XYZ', 'AOS', 'UPS', 'BLK', 'SW', 'PTC', 'TKO', 'HSIC', 'TEL', 'REG', 'ERIE', 'KO', 'CEG', 'VZ', 'PAYC', 'ON', 'STE', 'IQV', 'AMCR', 'CMS', 'MO', 'HOLX', 'MNST', 'FAST', 'ROST', 'DTE', 'PCG', 'DUK', 'SRE', 'TSN', 'AWK', 'MRK', 'WTW', 'MU', 'FANG', 'ADP', 'NOW', 'PLTR', 'ULTA', 'MMC', 'CAG', 'KHC', 'FIS', 'DLR', 'EQT', 'EMR', 'SNA', 'SYK', 'ALB', 'PANW', 'NDAQ', 'NSC', 'EVRG', 'XEL', 'FOXA', 'J', 'WDC', 'RCL', 'ORLY', 'APO', 'DVN', 'PPG', 'ITW', 'NFLX', 'GRMN', 'ICE', 'RJF', 'APP', 'LVS', 'PEP', 'EME', 'MOS',

In [ ]:
sp500df_ohlcv_append_flat = (
    sp500df_ohlcv_append_raw
    .stack(level=0, future_stack=True)
    .rename_axis(["date", "ticker"])
    .reset_index()
    .rename(columns={
        "Date": "date",
        "Ticker": "ticker",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume"
    })
    .query("date < @end")
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

sp500df_ohlcv_append_flat.columns.name = None

sp500df_ohlcv_append_flat.tail()

In [ ]:
sp500df_ohlcv_append_flat.to_sql("sp500_ohlcv", engine, if_exists="append", index=False)

In [22]:
sp500df = pd.read_sql("SELECT * FROM sp500_ohlcv;", engine, parse_dates=["date"]).pivot(index="date", columns="ticker", values=["open", "high", "low", "close", "adj_close", "volume"]).sort_index(axis=1, level=0)
sp500df.tail()

adj_close                                                  \
ticker               A        AAPL        ABBV        ABNB         ABT   
date                                                                     
2025-10-02  138.699997  257.130005  236.559998  121.489998  132.990005   
2025-10-03  141.639999  258.019989  233.910004  120.220001  134.589996   
2025-10-06  141.610001  256.690002  230.190002  120.349998  133.740005   
2025-10-07  138.559998  256.480011  232.830002  119.849998  133.020004   
2025-10-08  140.809998  258.059998  231.240005  119.989998  134.270004   

                                                                      ...  \
ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
date                                                                  ...   
2025-10-02  89.080002  244.339996  351.480011  241.669998  59.110001  ...   
2025-10-03  90.790001  245.320007  346.739990  241.990005  61.040001  ...   
2025-10-06  91.349998  248.169998  350.140015  242.500000  62.450001  ...   
2025-10-07  94.099998  251.229996  348.309998  233.750000  62.889999  ...   
2025-10-08  93.099998  252.979996  348.769989  237.929993  62.220001  ...   

               volume                                                          \
ticker             WY       WYNN        XEL         XOM        XYL        XYZ   
date                                                                            
2025-10-02  3592700.0  1286100.0  6982500.0  13059400.0  1347100.0  7852600.0   
2025-10-03  2943500.0  3619500.0  3850200.0  12948600.0  1245600.0  5755700.0   
2025-10-06  3680500.0  1799500.0  4082600.0  12034200.0  1330600.0  5318100.0   
2025-10-07  4592700.0  1443800.0  3715700.0  11945000.0  1190800.0  5428000.0   
2025-10-08  3475100.0  2316000.0  3142800.0  12299100.0  1029500.0  5993900.0   

                                                       
ticker            YUM        ZBH      ZBRA        ZTS  
date                                                   
2025-10-02  1800700.0   730500.0  450100.0  3262300.0  
2025-10-03  1288500.0   896200.0  452300.0  2569700.0  
2025-10-06  1395000.0  1065500.0  558200.0  3114900.0  
2025-10-07  1657100.0   920800.0  476800.0  2753700.0  
2025-10-08  1150600.0  1100700.0  577800.0  3016500.0  

[5 rows x 3048 columns]

In [23]:
daily_returns = sp500df["adj_close"].pct_change(fill_method=None).dropna(how="all")
daily_returns.tail()

ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
date,,,,,,,,,,,,,,,,,,,,,
2025-10-02,0.000866,0.006577,-0.031999,-0.006785,-0.003596,-0.013620,0.002585,0.022577,0.009988,-0.002363,...,-0.000802,0.009922,-0.008841,-0.006251,0.007641,0.046458,-0.012011,0.002831,0.013914,-0.003130
2025-10-03,0.021197,0.003461,-0.011202,-0.010454,0.012031,0.019196,0.004011,-0.013486,0.001324,0.032651,...,0.007621,-0.072596,0.008292,0.017702,0.005637,0.001823,-0.004361,0.016235,0.030962,-0.000478
2025-10-06,-0.000212,-0.005155,-0.015904,0.001081,-0.006315,0.006168,0.011617,0.009806,0.002108,0.023100,...,-0.006369,0.007521,0.009220,0.008299,0.000334,0.010786,-0.012675,-0.019349,-0.010065,-0.007239
2025-10-07,-0.021538,-0.000818,0.011469,-0.004155,-0.005384,0.030104,0.012330,-0.005227,-0.036082,0.007046,...,-0.021635,-0.013484,0.010494,0.000525,-0.011808,0.015942,-0.013846,0.002327,-0.017553,-0.017818
2025-10-08,0.016238,0.006160,-0.006829,0.001168,0.009397,-0.010627,0.006966,0.001321,0.017882,-0.010653,...,0.011466,-0.010902,0.000000,-0.002101,0.000405,0.026449,-0.004703,-0.007066,0.035868,0.005043


In [24]:
returns_1d = daily_returns.iloc[-1].dropna().sort_values(ascending=False)
returns_1d.head(10), returns_1d.tail(10)

(ticker
 AMD     0.113706
 DELL    0.090542
 ANET    0.083075
 SMCI    0.065553
 TYL     0.065551
 DDOG    0.062128
 MU      0.058431
 ON      0.056259
 FCX     0.053084
 CRWD    0.052268
 Name: 2025-10-08 00:00:00, dtype: float64,
 ticker
 MGM    -0.024155
 HBAN   -0.025060
 HCA    -0.025824
 TSN    -0.033020
 REGN   -0.033228
 LYV    -0.034636
 ARE    -0.036752
 CTVA   -0.038104
 WBD    -0.038172
 FICO   -0.098183
 Name: 2025-10-08 00:00:00, dtype: float64)

In [32]:
lookback_map = {
    "1D": 1,
    "1W": 5,
    "1M": 21,
    "3M": 63,
    "6M": 126
}

choice = input("Enter lookback horizon (1D, 1W, 1M, 3M, 6M, ALL, or number of trading days): ").upper()

if choice in lookback_map:
    window = lookback_map[choice]
    returns = (
        sp500df["adj_close"]
        .pct_change(window, fill_method=None)
        .iloc[-1]
        .dropna()
        .sort_values(ascending=False)
    )
    display(returns.head(10), returns.tail(10))

elif choice == "ALL":
    full_cum_returns = (
        sp500df["adj_close"].iloc[-1] / sp500df["adj_close"].iloc[0] - 1
    ).dropna().sort_values(ascending=False)
    display(full_cum_returns.head(10), full_cum_returns.tail(10))

elif choice.isdigit():
    window = int(choice)
    if len(sp500df) > window:
        returns = (
            sp500df["adj_close"]
            .pct_change(window, fill_method=None)
            .iloc[-1]
            .dropna()
            .sort_values(ascending=False)
        )
        display(returns.head(10), returns.tail(10))
    else:
        print(f"Not enough rows in dataset for {window} trading days.")

else:
    print("Invalid choice. Please enter 1D, 1W, 1M, 3M, 6M, ALL, or a number of trading days.")

ticker
AMD     0.113706
DELL    0.090542
ANET    0.083075
SMCI    0.065553
TYL     0.065551
DDOG    0.062128
MU      0.058431
ON      0.056259
FCX     0.053084
CRWD    0.052268
Name: 2025-10-08 00:00:00, dtype: float64

ticker
MGM    -0.024155
HBAN   -0.025060
HCA    -0.025824
TSN    -0.033020
REGN   -0.033228
LYV    -0.034636
ARE    -0.036752
CTVA   -0.038104
WBD    -0.038172
FICO   -0.098183
Name: 2025-10-08 00:00:00, dtype: float64

In [ ]:
fig = go.Figure()

for ticker in daily_returns.columns:
    fig.add_trace(go.Scatter(
        x=daily_returns.index,
        y=daily_returns[ticker],
        mode='lines',
        name=ticker,
        customdata=[[ticker]] * len(daily_returns),
        hovertemplate=(
        "Date: %{x}<br>" +
        "Return: %{y:.4f}<br>" +
        "Ticker: %{customdata[0]}<extra></extra>"
        )
    ))

fig.update_layout(
    title='Daily Returns of S&P 500 Tickers',
    xaxis_title='Date',
    yaxis_title='Daily Return',
    template='plotly_dark',
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show(config={'displaylogo': False})

In [67]:
user_input_stock_symbol = input("Enter stock symbol to inspect (e.g., 'AAPL', 'MSFT'): ").upper()

open = sp500df["open"][user_input_stock_symbol]
high = sp500df["high"][user_input_stock_symbol]
low = sp500df["low"][user_input_stock_symbol]
close = sp500df["close"][user_input_stock_symbol]
volume = sp500df["volume"][user_input_stock_symbol]
adj_close = sp500df["adj_close"][user_input_stock_symbol]

ticker = pd.DataFrame({
    "date": adj_close.index,
    "open": open.values,
    "high": high.values,
    "low": low.values,
    "close": close.values,
    "adj_close": adj_close.values,
    "volume": volume.values,
})

ticker.loc[:, "returns"] = ticker["adj_close"].pct_change(fill_method=None)
ticker.reset_index(drop=True, inplace=True)
ticker.tail()

,date,open,high,low,close,adj_close,volume,returns
187,2025-10-02,78.63999938964844,78.88999938964844,77.54000091552734,78.18000030517578,78.18000030517578,"9,263,900.0",-0.006228522628375477
188,2025-10-03,78.44999694824219,81.36000061035156,77.6500015258789,80.05999755859375,80.05999755859375,"12,115,900.0",0.024047035636727943
189,2025-10-06,80.1500015258789,82.37000274658203,80.1500015258789,82.11000061035156,82.11000061035156,"15,762,700.0",0.02560583455248633
190,2025-10-07,83.12999725341797,84.61000061035156,82.41000366210938,83.20999908447266,83.20999908447266,"16,109,700.0",0.013396644330099017
191,2025-10-08,83.69999694824219,84.38999938964844,82.87000274658203,84.04000091552734,84.04000091552734,"13,772,800.0",0.009974784763692846


In [68]:
lookback_map = {
    "1D": 1,
    "1W": 5,
    "1M": 21,
    "3M": 63,
    "6M": 126
}

choice = input("Enter lookback horizon (1D, 1W, 1M, 3M, 6M, ALL, or number of trading days): ").upper()

if choice in lookback_map:
    window = lookback_map[choice]
    if len(ticker) > window:
        ret = ticker["adj_close"].iloc[-1] / ticker["adj_close"].iloc[-window-1] - 1
        print(f"{choice} return for {user_input_stock_symbol}: {ret:.2%}")
    else:
        print(f"Not enough data for {choice}")

elif choice == "ALL":
    ret = ticker["adj_close"].iloc[-1] / ticker["adj_close"].iloc[0] - 1
    print(f"Total return for {user_input_stock_symbol}: {ret:.2%}")

elif choice.isdigit():
    window = int(choice)
    if len(ticker) > window:
        ret = ticker["adj_close"].iloc[-1] / ticker["adj_close"].iloc[-window-1] - 1
        print(f"{window}-day return for {user_input_stock_symbol}: {ret:.2%}")
    else:
        print(f"Not enough data for {window} trading days")

else:
    print("Invalid choice. Please enter 1D, 1W, 1M, 3M, 6M, ALL, or a number of trading days.")

150-day return for NEE: 20.20%


In [69]:
fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=ticker["date"], open=ticker["open"], high=ticker["high"], low=ticker["low"], close=ticker["close"], name="Candlestick"
))

fig.add_trace(go.Bar(
    x=ticker["date"], y=ticker["volume"], name="Volume", marker=dict(color="gray"), opacity=0.3, yaxis="y2"
))

fig.update_layout(
    title=f"{user_input_stock_symbol} Candlestick and Volume",
    xaxis=dict(title="Date", tickformat="%b %d"),
    yaxis=dict(title="Price ($)"),
    yaxis2=dict(title="Volume", overlaying="y", side="right", showgrid=False, color="gray"),
    legend=dict(x=0.01, y=0.99, bordercolor="black", borderwidth=1),
    bargap=0,
    template="plotly_dark",
    width=1200,
    height=500
)

fig.show(config={'displaylogo': False})

In [39]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"{user_input_stock_symbol} Daily Returns 2025",
    f"{user_input_stock_symbol} Daily Returns Distribution"
))

fig.add_trace(
    go.Scatter(x=ticker["date"], y=ticker["returns"], mode="lines+markers",
               line=dict(color="blue", width=0.5),
               marker=dict(symbol="triangle-down", size=4),
               name="Daily Returns"),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=ticker["returns"].dropna(), nbinsx=25,
                 marker=dict(color="blue", line=dict(color="black", width=1)),
                 opacity=0.7, name="Distribution"),
    row=1, col=2
)

fig.update_layout(
    template="plotly_dark",
    width=1200, height=500,
    showlegend=True
)

fig.update_xaxes(title_text="Date", tickformat="%b %d", row=1, col=1)
fig.update_yaxes(title_text="% Change", row=1, col=1)

fig.update_xaxes(title_text="Daily Return (% Change)", row=1, col=2)
fig.update_yaxes(title_text="Frequency", row=1, col=2)

fig.show(config={'displaylogo': False})

In [ ]:
fasb_fetch_year = datetime.datetime.now().year
url = f"https://xbrl.fasb.org/us-gaap/{fasb_fetch_year}/us-gaap-{fasb_fetch_year}.zip"

resp = requests.get(url)
resp.raise_for_status()

elements = []

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    xsd_files = [name for name in z.namelist() if name.startswith(f"us-gaap-{fasb_fetch_year}/elts/") and name.endswith(".xsd")]
    
    for xsd_file in xsd_files:
        with z.open(xsd_file) as f:
            try:
                tree = ET.parse(f)
                root = tree.getroot()
                for elem in root.findall(".//{http://www.w3.org/2001/XMLSchema}element"):
                    elements.append({
                        "id": elem.get("id"),
                        "name": elem.get("name"),
                        "type": elem.get("type"),
                        "substitutionGroup": elem.get("substitutionGroup"),
                        "balance": elem.get("balance"),
                        "periodType": elem.get("periodType"),
                        "source": xsd_file
                    })
            except Exception:
                # Skip non-XML or problematic files
                continue

xbrl_taxonomy = pd.DataFrame(elements).drop_duplicates(subset=["id"])

In [ ]:
xbrl_taxonomy.to_sql("xbrl_taxonomy", engine, if_exists="replace", index=False)

In [40]:
xbrl_taxonomy = pd.read_sql("SELECT * FROM xbrl_taxonomy;", engine)
xbrl_taxonomy

,id,name,type,substitutionGroup,balance,periodType,source
0,us-gaap_AccidentAndHealthInsuranceSegmentMember,AccidentAndHealthInsuranceSegmentMember,dtr-types:domainItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
1,us-gaap_OtherAccountsPayableAndAccruedLiabilities,OtherAccountsPayableAndAccruedLiabilities,xbrli:monetaryItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
2,us-gaap_AccountingForCertainLoansAndDebtSecuri...,AccountingForCertainLoansAndDebtSecuritiesAcqu...,dtr-types:textBlockItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
3,us-gaap_InterestsContinuedToBeHeldByTransferor...,InterestsContinuedToBeHeldByTransferorInFinanc...,xbrli:stringItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
4,us-gaap_InterestsContinuedToBeHeldByTransferor...,InterestsContinuedToBeHeldByTransferorInFinanc...,dtr-types:textBlockItemType,xbrli:item,None,None,us-gaap-2025/elts/us-gaap-2025.xsd
...,...,...,...,...,...,...,...
17350,tin-part_URI,URI,xs:anyURI,link:part,None,None,us-gaap-2025/elts/us-parts-tin-2025.xsd
17351,tin-part_Source_ASU_Number,Source_ASU_Number,tin-part:AsuNumber,link:part,None,None,us-gaap-2025/elts/us-parts-tin-2025.xsd
17352,tin-part_inlineURI,inlineURI,xs:anyURI,link:part,None,None,us-gaap-2025/elts/us-parts-tin-2025.xsd
17353,tin-part_pdfURI,pdfURI,xs:anyURI,link:part,None,None,us-gaap-2025/elts/us-parts-tin-2025.xsd


In [41]:
xbrl_query_ids = [
    "us-gaap_Assets",
    "us-gaap_AssetsCurrent",
    "us-gaap_AssetsNoncurrent",

    "us-gaap_Liabilities",
    "us-gaap_LiabilitiesCurrent",
    "us-gaap_LiabilitiesNoncurrent",

    "us-gaap_StockholdersEquity",
    "us-gaap_StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    "us-gaap_LiabilitiesAndStockholdersEquity",

    "us-gaap_RevenueFromContractWithCustomerExcludingAssessedTax",
    "us-gaap_SalesRevenueNet",
    "us-gaap_Revenues",
    "us-gaap_RevenueFromContractWithCustomerIncludingAssessedTax",

    "us-gaap_GrossProfit",
    "us-gaap_GrossProfitAbstract",

    "us-gaap_OperatingIncomeLoss",

    "us-gaap_NetIncomeLoss",
    "us-gaap_ProfitLoss",
    "us-gaap_NetIncomeLossAvailableToCommonStockholdersBasic",
    "us-gaap_NetIncomeLossAvailableToCommonStockholdersDiluted",
    "us-gaap_NetIncomeLossAttributableToNoncontrollingInterest",
    "us-gaap_NetIncomeLossAttributableToParent",
    "us-gaap_NetIncomeLossAttributableToRedeemableNoncontrollingInterest",
    "us-gaap_NetIncomeLossAttributableToNonredeemableNoncontrollingInterest",
    "us-gaap_NetIncomeLossIncludingPortionAttributableToNoncontrollingInterest",
    "us-gaap_NetIncomeLossAbstract",

    "us-gaap_EarningsPerShareBasic",
    "us-gaap_EarningsPerShareDiluted",

    "us-gaap_NetCashProvidedByUsedInOperatingActivities",
    "us-gaap_NetCashProvidedByUsedInInvestingActivities",
    "us-gaap_NetCashProvidedByUsedInFinancingActivities",
    "us-gaap_PaymentsToAcquirePropertyPlantAndEquipment",

    "us-gaap_WeightedAverageNumberOfSharesOutstandingBasic",
    "us-gaap_WeightedAverageNumberOfDilutedSharesOutstanding"
]

In [ ]:
with engine.connect() as conn:
    last_filed = pd.read_sql(
        "SELECT MAX(filed) as max_date FROM sp500_edgar_financials", conn
    ).iloc[0]["max_date"]

last_filed = pd.to_datetime(last_filed, errors="coerce").normalize()

id_to_name = dict(zip(xbrl_taxonomy["id"], xbrl_taxonomy["name"]))

sp500_edgar_financials_append = []

pbar = tqdm(sp500_reference.iterrows(), total=len(sp500_reference), desc="Updating financial metrics")

for _, row in pbar:
    ticker = row["ticker"]
    cik = str(row["cik"]).zfill(10)
    
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    r = requests.get(url, headers={"User-Agent": "dbater1993@gmail.com"})
    if r.status_code != 200:
        continue
    
    financial_metrics = r.json().get("facts", {}).get("us-gaap", {})
    
    for xid in xbrl_query_ids:
        xname = id_to_name.get(xid, xid.replace("us-gaap_", ""))
        
        metric_data = financial_metrics.get(xname)
        if not metric_data:
            continue
        
        for unit, datapoints in metric_data.get("units", {}).items():
            for dp in datapoints:
                filed_date = pd.to_datetime(dp.get("filed"), errors="coerce").normalize()
                end_date = pd.to_datetime(dp.get("end"), errors="coerce").normalize()
                
                if pd.isna(filed_date):
                    continue  # skip if no valid filed date
                
                if pd.isna(last_filed) or filed_date > last_filed:
                    sp500_edgar_financials_append.append({
                        "ticker": ticker,
                        "cik": cik,
                        "id": xid,
                        "metric": xname,
                        "unit": unit,
                        "value": dp.get("val"),
                        "fy": dp.get("fy"),
                        "fp": dp.get("fp"),
                        "form": dp.get("form"),
                        "filed": filed_date,
                        "end": end_date
                    })
    
    pbar.set_description(f"Collected: {len(sp500_edgar_financials_append)} new rows")
    time.sleep(0.2)

pbar.close()

sp500_edgar_financials_append

In [ ]:
sp500_edgar_financials_append = pd.DataFrame(sp500_edgar_financials_append)
sp500_edgar_financials_append.to_sql("sp500_edgar_financials", engine, if_exists="append", index=False)

In [42]:
user_stock_fundamentals_query = f"""
SELECT "end", id, metric, value
FROM sp500_edgar_financials
WHERE ticker = '{user_input_stock_symbol}' AND form = '10-K'
ORDER BY "end";
"""
user_stock_fundamentals_query_df = pd.read_sql(user_stock_fundamentals_query, engine)
user_stock_fundamentals_query_df

,end,id,metric,value
0,2006-12-31,us-gaap_StockholdersEquity,StockholdersEquity,9.930000e+09
1,2007-12-31,us-gaap_WeightedAverageNumberOfSharesOutstandi...,WeightedAverageNumberOfSharesOutstandingBasic,3.977000e+08
2,2007-12-31,us-gaap_NetCashProvidedByUsedInOperatingActivi...,NetCashProvidedByUsedInOperatingActivities,3.593000e+09
3,2007-12-31,us-gaap_NetCashProvidedByUsedInInvestingActivi...,NetCashProvidedByUsedInInvestingActivities,-4.578000e+09
4,2007-12-31,us-gaap_NetCashProvidedByUsedInFinancingActivi...,NetCashProvidedByUsedInFinancingActivities,6.550000e+08
...,...,...,...,...
1219,2024-12-31,us-gaap_Assets,Assets,1.901440e+11
1220,2024-12-31,us-gaap_AssetsCurrent,AssetsCurrent,1.195100e+10
1221,2024-12-31,us-gaap_EarningsPerShareBasic,EarningsPerShareBasic,3.380000e+00
1222,2024-12-31,us-gaap_ProfitLoss,ProfitLoss,5.698000e+09


In [43]:
user_stock_fundamentals_query_df["end"] = pd.to_datetime(user_stock_fundamentals_query_df["end"])

pivot = user_stock_fundamentals_query_df.pivot_table(
    index="metric",
    columns=user_stock_fundamentals_query_df["end"].dt.year,
    values="value",
    aggfunc="last"
)

growth = (pivot - pivot.shift(axis=1)) / pivot.shift(axis=1).abs()

growth_clipped = growth.clip(lower=-1, upper=1) * 100   # -100% to +100%

metrics = pivot.index.tolist()
metrics_sorted = sorted([m for m in metrics if m != "Assets"])
if "Assets" in metrics:
    metrics_sorted = ["Assets"] + metrics_sorted

pivot = pivot.loc[metrics_sorted]
growth_clipped = growth_clipped.loc[metrics_sorted]

colorscale = [
    [0.00, "#800000"],
    [0.10, "#b2182b"],
    [0.20, "#d6604d"],
    [0.30, "#f4a582"],
    [0.40, "#fddbc7"],
    [0.50, "#ffffbf"],
    [0.60, "#d9ef8b"],
    [0.70, "#91cf60"],
    [0.80, "#4daf4a"],
    [0.90, "#1b7837"],
    [1.00, "#00441b"]
]

fig = go.Figure(
    data=go.Heatmap(
        z=growth_clipped.fillna(0).values,
        x=growth_clipped.columns,
        y=growth_clipped.index,
        colorscale=colorscale,
        zmin=-100,
        zmax=100,
        colorbar=dict(title="YoY % Growth"),
        customdata=pivot.fillna("").values,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Year: %{x}<br>"
            "Value: %{customdata:,}<br>"
            "Growth: %{z:.2f}%<extra></extra>"
        )
    )
)

fig.update_layout(
    title=f"{user_input_stock_symbol} Fundamentals Growth Heatmap (10-K)",
    xaxis_title="Year",
    yaxis_title="Metric",
    height=800,
    yaxis=dict(categoryorder="array", categoryarray=metrics_sorted),
    template="plotly_dark"
)

fig.update_yaxes(autorange="reversed")
fig.show(config={'displaylogo': False})

In [47]:
pd.options.display.float_format = '{:,}'.format

sectors_query = "SELECT DISTINCT sector FROM sp500_reference ORDER BY sector;"

latest_10k_query = """
WITH latest_end_per_ticker AS (
    SELECT
        ticker,
        MAX("end") AS latest_end
    FROM sp500_edgar_financials
    GROUP BY ticker
)
SELECT 
    f.ticker,
    f.metric,
    f.value,
    f.end
FROM sp500_edgar_financials f
JOIN latest_end_per_ticker l
    ON f.ticker = l.ticker
   AND f.end = l.latest_end
ORDER BY f.ticker, f.metric;
"""

latest_10k_df = pd.read_sql(latest_10k_query, engine)
latest_10k_df.loc[:, 'end'] = pd.to_datetime(latest_10k_df.loc[:, 'end'], errors='coerce')

latest_10k_df = latest_10k_df.merge(
    sp500_reference[['ticker', 'sector']],
    on='ticker',
    how='left'
)

latest_10k_df

,ticker,metric,value,end,sector
0,A,Assets,"12,226,000,000.0",2025-07-31 00:00:00,Health Care
1,A,AssetsCurrent,"4,253,000,000.0",2025-07-31 00:00:00,Health Care
2,A,EarningsPerShareBasic,3.05,2025-07-31 00:00:00,Health Care
3,A,EarningsPerShareBasic,1.18,2025-07-31 00:00:00,Health Care
4,A,EarningsPerShareDiluted,1.18,2025-07-31 00:00:00,Health Care
...,...,...,...,...,...
13418,ZTS,StockholdersEquityIncludingPortionAttributable...,"4,977,000,000.0",2025-06-30 00:00:00,Health Care
13419,ZTS,WeightedAverageNumberOfDilutedSharesOutstanding,"446,700,000.0",2025-06-30 00:00:00,Health Care
13420,ZTS,WeightedAverageNumberOfDilutedSharesOutstanding,"445,500,000.0",2025-06-30 00:00:00,Health Care
13421,ZTS,WeightedAverageNumberOfSharesOutstandingBasic,"446,300,000.0",2025-06-30 00:00:00,Health Care


In [66]:
sp500df_flattened_adj_close = (
    sp500df
    .loc[:, 'adj_close']
    .stack()
    .copy()
    .reset_index()
    .rename(columns={'date': 'date', 'ticker': 'ticker', 0: 'adj_close'})
    .sort_values(['ticker', 'date'])
    .reset_index(drop=True)
)

sp500df_flattened_adj_close_last_two_days = (
    sp500df
    .loc[:, 'adj_close']
    .stack()
    .copy()
    .reset_index()
    .rename(columns={'date': 'date', 'ticker': 'ticker', 0: 'adj_close'})
    .sort_values(['ticker', 'date'])
    .groupby('ticker', group_keys=False)
    .tail(2)
    .reset_index(drop=True)
)
sp500df_flattened_adj_close_last_two_days

,date,ticker,adj_close
0,2025-10-07,A,138.55999755859375
1,2025-10-08,A,140.80999755859375
2,2025-10-07,AAPL,256.4800109863281
3,2025-10-08,AAPL,258.05999755859375
4,2025-10-07,ABBV,232.8300018310547
...,...,...,...
1011,2025-10-08,ZBH,98.36000061035156
1012,2025-10-07,ZBRA,296.6400146484375
1013,2025-10-08,ZBRA,307.2799987792969
1014,2025-10-07,ZTS,142.77000427246094


In [ ]:
sp500df_flat = (
    sp500df.loc[:, 'adj_close']
    .stack()
    .reset_index()
    .rename(columns={
        'date': 'date',
        'ticker': 'ticker',
        0: 'adj_close'
    })
    .sort_values(['ticker', 'date'])
    .groupby('ticker', group_keys=False)
    .tail(2)
    .reset_index(drop=True)
    .copy()
)

mask = latest_10k_df.loc[:, 'metric'].str.strip().str.lower() == \
       "weightedaveragenumberofdilutedsharesoutstanding"

shares_outstanding = latest_10k_df.loc[mask, ['ticker', 'value', 'sector']].copy()

merged = sp500df_flat.merge(shares_outstanding, on='ticker', how='inner')

merged.loc[:, 'market_cap'] = merged['adj_close'] * merged['value']

merged = (
    merged.sort_values(['ticker', 'date'])
    .assign(pct_change=lambda df: df.groupby('ticker')['market_cap'].pct_change())
)

marketcap_latest = (
    merged.groupby('ticker', as_index=False)
    .last()
    .dropna(subset=['pct_change'])
    .copy()
)

marketcap_latest

c:\Users\dbate\Desktop\venv\Lib\site-packages\pandas\core\frame.py:5239: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy




,ticker,date,adj_close,value,sector,market_cap,pct_change
0,A,2025-10-08,140.80999755859375,"285,000,000.0",Health Care,"40,130,849,304.19922",0.0
1,AAPL,2025-10-08,258.05999755859375,"15,051,726,000.0",Information Technology,"3,884,248,374,812.622",0.006927064493942758
2,ABBV,2025-10-08,231.24000549316406,"1,772,000,000.0",Health Care,"409,757,289,733.8867",0.000564652738565874
3,ABNB,2025-10-08,119.98999786376953,"626,000,000.0",Consumer Discretionary,"75,113,738,662.71973",-0.004769475357710662
4,ABT,2025-10-08,134.27000427246094,"1,749,054,000.0",Health Care,"234,845,488,052.7649",-0.0010172289222000197
...,...,...,...,...,...,...,...
484,XYZ,2025-10-08,81.11000061035156,"618,928,000.0",Financials,"50,201,250,457.76367",-0.013036136009555
485,YUM,2025-10-08,146.02999877929688,"282,000,000.0",Consumer Discretionary,"41,180,459,655.76172",0.003558718861210064
486,ZBH,2025-10-08,98.36000061035156,"198,300,000.0",Health Care,"19,504,788,121.032715",-0.0035175879396984744
487,ZBRA,2025-10-08,307.2799987792969,"51,282,273.0",Information Technology,"15,758,016,784.83957",-0.005124255986013404


In [52]:
root_name = "S&P 500"

root_df = pd.DataFrame({
    "labels": [root_name],
    "parents": [""],
    "values": [marketcap_latest["market_cap"].sum()],
    "pct_change": [0],
    "adj_close": [np.nan],
    "shares": [np.nan],
})

sector_df = (
    marketcap_latest.groupby("sector", as_index=False)
    .agg({"market_cap": "sum"})
    .assign(parents=root_name, pct_change=0, adj_close=np.nan, shares=np.nan)
    .rename(columns={"sector": "labels", "market_cap": "values"})
)

company_df = marketcap_latest.rename(
    columns={
        "ticker": "labels",
        "sector": "parents",
        "market_cap": "values",
        "adj_close": "adj_close",
        "value": "shares"
    }
)[["labels", "parents", "values", "pct_change", "adj_close", "shares"]]

tree_data = pd.concat([root_df, sector_df, company_df], ignore_index=True)

fig = go.Figure(
    go.Treemap(
        labels=tree_data["labels"],
        parents=tree_data["parents"],
        values=tree_data["values"],
        customdata=tree_data[["adj_close", "shares"]],
        marker=dict(
            colors=tree_data["pct_change"],
            colorscale=[(0, "red"), (0.5, "lightgray"), (1, "green")],
            cmin=-0.05,
            cmax=0.05,
            showscale=True,
            colorbar=dict(title="% Change", tickformat=".2%")
        ),
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Parent: %{parent}<br>"
            "Market Cap: $%{value:,.0f}<br>"
            "Adj Close: $%{customdata[0]:,.2f}<br>"
            "Shares: %{customdata[1]:,.0f}<br>"
            "% Change: %{color:.2%}<extra></extra>"
        ),
        branchvalues="total",
    )
)

latest_date = marketcap_latest["date"].max().strftime("%B %d, %Y")
fig.update_layout(
    title=f"S&P 500 Market Cap Heatmap – {latest_date}",
    template="plotly_dark",
    height=900,
    margin=dict(t=80, l=40, r=40, b=40),
)

fig.show(config={"displaylogo": False})

In [ ]:
revenue_summary = (
    latest_10k_df[latest_10k_df['metric'].str.lower() == "earningspersharediluted"]
    .groupby('sector')['value']
    .agg(['count', 'sum', 'mean', 'median'])
    .sort_values('mean', ascending=False)
    .reset_index()
)
revenue_summary

In [ ]:
sector_name = input("Enter sector name (e.g., 'Information Technology'): ").title()

sector_metric_summary = (
    latest_10k_df[latest_10k_df['sector'] == sector_name]
    .groupby('metric')['value']
    .agg(['count', 'sum', 'mean', 'median'])
    .sort_values('mean', ascending=False)
    .reset_index()
)

sector_metric_summary

In [ ]:
sector_breakdown = (
    sp500_reference
    .groupby('sector')['ticker']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='count')
)

sector_breakdown

In [ ]:
fama_french_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip'

response = requests.get(fama_french_url)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    file_name = z.namelist()[0]
    with z.open(file_name) as file:
        fama_french_five_factor = pd.read_csv(file,index_col=0, parse_dates=True,skiprows=3)

fama_french_five_factor = fama_french_five_factor.iloc[:-1]

fama_french_five_factor.reset_index(inplace=True)

fama_french_five_factor.columns = ['Date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']

fama_french_five_factor['Date'] = pd.to_datetime(fama_french_five_factor['Date'])

fama_french_five_factor = fama_french_five_factor[
    (fama_french_five_factor['Date'] >= pd.to_datetime(ohlcv_start).normalize()) &
    (fama_french_five_factor['Date'] <= pd.to_datetime(ohlcv_end).normalize())
]

fama_french_five_factor.reset_index(drop=True, inplace=True)

fama_french_five_factor

In [ ]:
daily_returns.index = pd.to_datetime(daily_returns.index)

train_start = fama_french_five_factor['Date'].min()
train_end = fama_french_five_factor['Date'].max()

excess_returns = daily_returns.loc[
    daily_returns.index.intersection(fama_french_five_factor['Date'])
].sub(
    fama_french_five_factor.set_index("Date").loc[daily_returns.index.intersection(fama_french_five_factor['Date']), "RF"] / 100,
    axis=0
)

train_returns = excess_returns.loc[train_start:train_end]
train_factors = fama_french_five_factor.set_index("Date").loc[train_start:train_end, ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]

regression_results = []

for ticker in train_returns.columns:
    y_train = train_returns[ticker]
    X_train = sm.add_constant(train_factors)

    common_idx = y_train.index.intersection(X_train.index)
    y_train = y_train.loc[common_idx]
    X_train = X_train.loc[common_idx]

    model = sm.OLS(y_train, X_train).fit()

    result = {
        'Ticker': ticker,
        'α': model.params['const'],
        'Mkt-RF': model.params.get('Mkt-RF', None),
        'SMB': model.params.get('SMB', None),
        'HML': model.params.get('HML', None),
        'RMW': model.params.get('RMW', None),
        'CMA': model.params.get('CMA', None),
        'R-squared': model.rsquared
    }
    regression_results.append(result)

regression_summary_df = pd.DataFrame(regression_results)

regression_summary_df.sort_values(by='α', ascending=False).reset_index(drop=True).head(25).round(4)


In [ ]:
def regression_sort():
    sort_by = input(f"Choose a column to sort by {list(regression_summary_df.columns[1:])}: ")
    order = input("Sort ascending? (yes/no): ").lower() == "yes"
    return regression_summary_df.sort_values(by=sort_by, ascending=order).reset_index(drop=True).head(10)

In [ ]:
y_actual = train_returns[user_input_stock_symbol]
X = sm.add_constant(train_factors)
common_idx = y_actual.index.intersection(X.index)
y_actual = y_actual.loc[common_idx]
X = X.loc[common_idx]
model = sm.OLS(y_actual, X).fit()
y_pred = model.predict(X)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y_actual.index,
    y=y_actual,
    mode='lines',
    name='Actual Excess Return',
    line=dict(width=1.5),
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Return: %{y:.4f}<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=y_actual.index,
    y=y_pred,
    mode='lines',
    name='Predicted Excess Return (Line of Best Fit)',
    line=dict(width=2, dash='dash'),
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Predicted: %{y:.4f}<extra></extra>'
))

fig.update_layout(
    title=f"{user_input_stock_symbol}: Actual vs Predicted Excess Returns",
    xaxis_title="Date",
    yaxis_title="Excess Return",
    template="plotly_dark",
    height=600,
    width=1200,
    legend=dict(
        x=0.01, y=0.99,
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(255,255,255,0.1)'
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor='rgba(128,128,128,0.2)',
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=list([
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=12, label="1Y", step="month", stepmode="backward"),
                dict(step="all", label="All")
            ])
        )
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='rgba(128,128,128,0.2)'
    )
)

fig.show(config={'displaylogo': False})